# 2. Reproduce the tobacco detector experiments

**Goal:** train YOLOv8n, YOLOv8m, and YOLOv8x on the original frozen split and evaluate their best checkpoints on validation and test images.

By default, this notebook only inspects settings and explains missing inputs. To train, install `requirements-experiments.txt` in your notebook environment, provide the original dataset, confirm its label meaning, and set `RUN_TRAINING = True` below. Initial pretrained `.pt` weights may be downloaded by Ultralytics, so the first training run may need internet access. Fine-tuned historical weights are not needed to retrain from pretrained models.

Use the README for environment setup. The original run used Windows 11 and an RTX 5070 Ti. A CUDA-compatible PyTorch installation is required for the default GPU device. CPU runs are possible but will be much slower and must not be compared as GPU timing.


In [ ]:
from pathlib import Path
import sys
import json
import os
import tempfile

# Find the checkout whether Jupyter starts at its root or inside notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/run_yolo_experiments.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from this repository or its notebooks directory.')
sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import run_script, new_run, frozen_assignments, write_local_splits, check_detector_results

# Caches are local runtime files, not historical experiment artifacts.
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'tobacco-notebook-mpl'))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'experiments/.ultralytics'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
RESULTS = ROOT / 'experiments/results'
print('Repository:', ROOT)


## Choose inputs and a new output directory

Provide the original dataset in this layout:

```text
tobacco_yolo/
  images/<image-name>.jpg
  labels/<same-image-name>.txt
```

Each label row is `class_id x_center y_center width height`, with normalized coordinates. **Class 0 must mean tobacco**, not weeds. Software can check IDs but cannot determine their semantic meaning; confirm it from the annotation source.

Every new run uses a fresh directory under `experiments/reproductions/`. Reusing a run name raises an error to protect previous results. To retry a failed experiment, choose another name. No historical result is overwritten.


In [ ]:
RUN_TRAINING = False
CLASS_ZERO_CONFIRMED_TOBACCO = False
DATASET = ROOT / 'data/tobacco_yolo'  # Or Path('/your/external/tobacco_yolo')
RUN_NAME = 'detector-reproduction-01'
DEVICE = '0'  # CUDA GPU index; use 'cpu' only for an explicitly CPU experiment.
MODELS = ['yolov8n.pt', 'yolov8m.pt', 'yolov8x.pt']
EPOCHS, IMAGE_SIZE, BATCH_SIZE = 50, 1024, 14
run_dir = None
training_complete = False
print('Dataset present:', (DATASET / 'images').is_dir() and (DATASET / 'labels').is_dir())
print('Training enabled:', RUN_TRAINING)
display(pd.Series(json.loads((RESULTS / 'yolo/run_metadata.json').read_text())))


## Preserve the original split

The saved manifests contain Windows paths. We read their filenames portably, verify them against `items.csv`, and later write absolute paths for this machine. We **do not generate a new random split**. Expected membership is 78 train, 16 validation, 16 test.

The split seed `20260627` controls dataset splitting; it is not a recorded training seed. The original runner relies on framework defaults for training randomness and augmentations. Its full resolved training arguments are not included here. Capturing the new environment makes differences auditable, but exact numerical replication is not guaranteed.

The saved manifest digest includes path strings and box counts, not image bytes. It is not a dataset-content checksum. Counts and filenames cannot establish that two datasets are identical; obtain archive checksums from the dataset owner when available. The split is image-based and does not establish independence by field or acquisition session.


In [ ]:
assignments = frozen_assignments(ROOT)
display(pd.DataFrame({'split': assignments.keys(), 'images': [len(v) for v in assignments.values()]}))


## Validate the data and capture the machine

When training is enabled, this cell checks the hardware, runs the existing preflight script, verifies the historical counts and per-image box counts, and relocates the manifests. The preflight report and environment files go to the new run directory.

A mismatch stops the reproduction. If you intentionally use different data, that is a new experiment and needs its own split protocol rather than these historical manifests.


In [ ]:
if RUN_TRAINING:
    if not CLASS_ZERO_CONFIRMED_TOBACCO:
        raise ValueError('Confirm class 0 means tobacco before enabling training.')
    DATASET = DATASET.expanduser().resolve()
    if not (DATASET / 'images').is_dir() or not (DATASET / 'labels').is_dir():
        raise FileNotFoundError('Supply the external images/ and labels/ directories first.')
    import torch
    if DEVICE != 'cpu' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Install a compatible PyTorch/driver environment first.')
    run_dir = new_run(ROOT, RUN_NAME)
    run_script(ROOT, 'preflight_dataset.py', '--dataset', DATASET,
               '--output', run_dir / 'dataset_preflight.json', '--expected-class-zero', 'tobacco')
    preflight = json.loads((run_dir / 'dataset_preflight.json').read_text())
    assert preflight['status'] == 'ok'
    assert preflight['images'] == 110 and preflight['boxes'] == 4436
    assert preflight['extra_label_files_count'] == 0
    assert set(preflight['class_counts']) == {'0'}, 'Expected single-class tobacco annotations.'
    items = pd.read_csv(ROOT / 'experiments/splits/items.csv')
    for item in items.itertuples():
        label = DATASET / 'labels' / f'{item.stem}.txt'
        assert sum(bool(line.strip()) for line in label.read_text().splitlines()) == item.boxes
    local_splits = write_local_splits(ROOT, DATASET, run_dir / 'splits')
    run_script(ROOT, 'capture_environment.py', '--output', run_dir / 'environment')
else:
    print('Skipped data preflight: set DATASET, confirm class meaning, and enable RUN_TRAINING to continue.')


## Train and evaluate all three models

The existing runner uses SGD, initial learning rate 0.005, momentum 0.937, weight decay 0.0005, cosine scheduling, and automatic mixed precision. It trains for 50 epochs at size 1024 and batch 14, then evaluates selected weights on both validation and test splits.

If the largest model runs out of GPU memory, lower `BATCH_SIZE` for **all three models**, choose a new run name, and rerun. That change must be reported as a deviation. Training duration depends on hardware; use a persistent kernel and allow the run to finish.

The script records failures in CSV even when its process exits successfully. The check below explicitly rejects failed training, failed evaluation, and missing metrics. Full model folders and weights remain outside Git.


In [ ]:
if RUN_TRAINING:
    if run_dir is None:
        raise RuntimeError('Run the data validation cell first.')
    run_script(ROOT, 'run_yolo_experiments.py', '--dataset', DATASET, '--splits', local_splits,
               '--models', *MODELS, '--epochs', EPOCHS, '--imgsz', IMAGE_SIZE,
               '--batch', BATCH_SIZE, '--device', DEVICE, '--output', run_dir / 'yolo',
               '--class-name', 'tobacco')
    check_detector_results(run_dir / 'yolo/model_comparison.csv')
    training_complete = True
else:
    print('Training skipped. Historical results remain available in notebook 1.')


## Compare the new measurements with the historical run

Differences can arise from framework versions, randomness, GPU kernels, and hardware. Do not substitute historical timing for the new machine's timing. Report changed settings together with the captured environment. Test data should not be used to tune hyperparameters.


In [ ]:
if training_complete:
    old = pd.read_csv(RESULTS / 'yolo/model_comparison.csv').set_index('model_name')
    new = pd.read_csv(run_dir / 'yolo/model_comparison.csv').set_index('model_name')
    keys = ['test_metrics/precision(B)', 'test_metrics/recall(B)',
            'test_metrics/mAP50(B)', 'test_metrics/mAP50-95(B)', 'test_speed_inference_ms']
    display(pd.concat({'Historical': old[keys], 'Reproduction': new[keys]}, axis=1))
    print('Outputs:', run_dir)
    print('Use this trained model in notebook 3:', run_dir / 'yolo/yolov8x/weights/best.pt')
else:
    print('No new experiment was run; there are no new metrics to compare.')
